# Metadata Enrichment

This notebook enriches the user-history dataset using product metadata from the Amazon Reviews 2023 Electronics dataset.

Only metadata corresponding to products appearing in the user histories or ground-truth interactions are retained. This avoids loading the complete metadata collection into memory, making the preprocessing pipeline scalable for large datasets.

The enriched dataset will be used in the next stage to generate synthetic natural language queries.

In [1]:
import json
from pathlib import Path

import pandas as pd
from tqdm import tqdm

pd.set_option("display.max_columns", None)
tqdm.pandas()

In [2]:
PROJECT_DIR= Path.home()/"query-conditioned-recommender"
RAW= PROJECT_DIR/"data"/"raw"
PROCESSED= PROJECT_DIR/"data"/"processed"
DATASET_PATH= PROCESSED/"user_histories.parquet"
META_PATH= RAW/"meta_Electronics.jsonl"
OUTPUT_PATH= PROCESSED/"enriched_user_histories.parquet"

In [3]:
dataset= pd.read_parquet(DATASET_PATH)
print(dataset.shape)
dataset.head()

(5559, 12)


,sample_id,user_id,history_asins,ground_truth_asin,history_length,history_timestamps,history_ratings,ground_truth_timestamp,ground_truth_rating,query,constraint_type,constraint_values
0,0,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B075B5T7L5, B00WA4PSEM, B09VS4V18K, B00G2UTVH...",B007VKII6A,5,"[2015-09-01T12:14:16.000000, 2015-09-01T12:23:...","[5, 5, 5, 5, 5]",2017-05-21 14:36:52.000,2,None,None,None
1,1,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B00WA4PSEM, B09VS4V18K, B00G2UTVH0, B07RS68PJ...",B07454F4JH,5,"[2015-09-01T12:23:56.000000, 2016-10-02T21:27:...","[5, 5, 5, 5, 2]",2017-08-24 17:49:29.844,5,None,None,None
2,2,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B09VS4V18K, B00G2UTVH0, B07RS68PJ7, B007VKII6...",B00SMVLPIK,5,"[2016-10-02T21:27:51.000000, 2016-10-02T22:15:...","[5, 5, 5, 2, 5]",2017-08-24 17:57:36.638,5,None,None,None
3,3,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B00G2UTVH0, B07RS68PJ7, B007VKII6A, B07454F4J...",B0179DDIQI,5,"[2016-10-02T22:15:43.000000, 2016-10-23T09:32:...","[5, 5, 2, 5, 5]",2017-08-24 18:06:02.421,5,None,None,None
4,4,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B07RS68PJ7, B007VKII6A, B07454F4JH, B00SMVLPI...",B00X088DZI,5,"[2016-10-23T09:32:31.000000, 2017-05-21T14:36:...","[5, 2, 5, 5, 5]",2019-05-11 17:22:30.699,5,None,None,None


In [4]:
needed_asins= set()
#History products
for history in dataset["history_asins"]:
    needed_asins.update(history)
#Ground-truth products
needed_asins.update(dataset["ground_truth_asin"].tolist())
print(f"Unique ASINs required: {len(needed_asins):,}")

Unique ASINs required: 6,585


In [5]:
metadata_lookup= {}
with open(META_PATH, "r") as f:
    for line in tqdm(f):
        record= json.loads(line)
        asin= record.get("parent_asin")
        if asin in needed_asins:
            metadata_lookup[asin]= {
                "title": record.get("title"),
                "categories": record.get("categories"),
                "description": record.get("description")
            }
print(f"Metadata found: {len(metadata_lookup):,}")

0it [00:00, ?it/s]

1610012it [00:29, 54911.43it/s]

Metadata found: 6,585


In [6]:
coverage= len(metadata_lookup)/len(needed_asins)
print(f"Metadata coverage: {coverage:.2%}")

Metadata coverage: 100.00%


In [7]:
def enrich_history(history):
    titles= []
    categories= []
    descriptions= []
    for asin in history:
        item= metadata_lookup.get(asin)
        if item is None:
            titles.append(None)
            categories.append(None)
            descriptions.append(None)
        else:
            titles.append(item["title"])
            categories.append(item["categories"])
            descriptions.append(item["description"])
    return pd.Series([titles,categories,descriptions])

In [8]:
dataset[["history_titles","history_categories","history_descriptions"]]= dataset["history_asins"].progress_apply(enrich_history)

  0%|          | 0/5559 [00:00<?, ?it/s]

100%|██████████| 5559/5559 [00:00<00:00, 18282.72it/s]


In [9]:
def enrich_ground_truth(asin):
    item= metadata_lookup.get(asin)
    if item is None:
        return pd.Series([None,None,None])
    return pd.Series([item["title"],item["categories"],item["description"]])

In [10]:
dataset[
    [
        "ground_truth_title",
        "ground_truth_categories",
        "ground_truth_description"
    ]
]= dataset["ground_truth_asin"].progress_apply(enrich_ground_truth)

100%|██████████| 5559/5559 [00:00<00:00, 18142.03it/s]


In [11]:
dataset.head()

,sample_id,user_id,history_asins,ground_truth_asin,history_length,history_timestamps,history_ratings,ground_truth_timestamp,ground_truth_rating,query,constraint_type,constraint_values,history_titles,history_categories,history_descriptions,ground_truth_title,ground_truth_categories,ground_truth_description
0,0,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B075B5T7L5, B00WA4PSEM, B09VS4V18K, B00G2UTVH...",B007VKII6A,5,"[2015-09-01T12:14:16.000000, 2015-09-01T12:23:...","[5, 5, 5, 5, 5]",2017-05-21 14:36:52.000,2,None,None,None,[Patriot LX Series 64GB High Speed Micro SDXC ...,"[[Electronics, Computers & Accessories, Comput...",[[Patriot’s LX Series UHS-I compatible MicroSD...,IDANCE CRAZY101 - Watt - Channel Recording Stu...,"[Electronics, Headphones, Earbuds & Accessorie...",[Super Aticulated Headband. Extra Bass with Op...
1,1,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B00WA4PSEM, B09VS4V18K, B00G2UTVH0, B07RS68PJ...",B07454F4JH,5,"[2015-09-01T12:23:56.000000, 2016-10-02T21:27:...","[5, 5, 5, 5, 2]",2017-08-24 17:49:29.844,5,None,None,None,[The AquaAudio Cubo – Waterproof Bluetooth Wir...,"[[Electronics, Portable Audio & Video, Portabl...",[[AquaAudio Cubo Bluetooth Wireless Speaker wi...,Seagate Backup Plus Portable 5TB External Hard...,"[Electronics, Computers & Accessories, Data St...","[Looking for easy, on-the-go storage? Seagate ..."
2,2,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B09VS4V18K, B00G2UTVH0, B07RS68PJ7, B007VKII6...",B00SMVLPIK,5,"[2016-10-02T21:27:51.000000, 2016-10-02T22:15:...","[5, 5, 5, 2, 5]",2017-08-24 17:57:36.638,5,None,None,None,[Western Digital 2TB WD Black Performance Inte...,"[[Electronics, Computers & Accessories, Data S...",[[WD Black performance storage is designed to ...,Panasonic FOLDZ On-Ear Stereo Headphones with ...,"[Electronics, Headphones, Earbuds & Accessorie...",[The Panasonic FOLDZ RP-DJS150M combines the h...
3,3,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B00G2UTVH0, B07RS68PJ7, B007VKII6A, B07454F4J...",B0179DDIQI,5,"[2016-10-02T22:15:43.000000, 2016-10-23T09:32:...","[5, 5, 2, 5, 5]",2017-08-24 18:06:02.421,5,None,None,None,[EBL Pack of 6 1000mAh NI-MH Home Phone Rechar...,"[[Electronics, Accessories & Supplies, Telepho...","[[features:, Chemistry: Ni-MH Voltage: 2.4V C...",BienSound HW50 Stereo Folding Headsets Strong ...,"[Electronics, Headphones, Earbuds & Accessorie...",[]
4,4,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,"[B07RS68PJ7, B007VKII6A, B07454F4JH, B00SMVLPI...",B00X088DZI,5,"[2016-10-23T09:32:31.000000, 2017-05-21T14:36:...","[5, 2, 5, 5, 5]",2019-05-11 17:22:30.699,5,None,None,None,"[CyberPower CP900AVR AVR UPS System, 900VA/560...","[[Electronics, Computers & Accessories, Comput...",[[A mini-tower UPS with line interactive topol...,Kastar 5-Pack HHR-P105 Type 31 Battery Replace...,"[Electronics, Accessories & Supplies, Telephon...",[Bullet Point 1 Kastar Brand Replacement NI-MH...


In [12]:
dataset.sample(5)

,sample_id,user_id,history_asins,ground_truth_asin,history_length,history_timestamps,history_ratings,ground_truth_timestamp,ground_truth_rating,query,constraint_type,constraint_values,history_titles,history_categories,history_descriptions,ground_truth_title,ground_truth_categories,ground_truth_description
230,230,AEBEVCYY3HASVJ2Z5Q2Q36TENT2A,"[B076751BN8, B088NGT57C, B071X9DRSB, B00GMCLLY...",B07H6LYZSD,5,"[2018-11-18T19:52:10.975000, 2019-01-26T21:31:...","[3, 5, 5, 3, 5]",2019-02-24 02:54:58.293,5,None,None,None,[Intel NCSM2450.DK1 Movidius Neural Compute St...,"[[Electronics, Computers & Accessories, Comput...",[[The Movidius Neural Compute Stick is a minia...,"USB C Hub, Aluminum USB C Adapter with PD Port...",[],[]
4808,4808,AHKLYY7DZ4RECGZXP6KUALGUFRJA,"[B00D38XF6M, B00FGI3CBU, B01FWT4V12, B012HGRTL...",B07BGW4D2Q,5,"[2015-05-30T17:53:48.000000, 2015-06-05T19:31:...","[3, 5, 5, 5, 5]",2019-08-30 19:11:12.771,5,None,None,None,"[Kindle Fire HDX 7"", HDX Display, Wi-Fi and 4G...","[[], [Electronics, Computers & Accessories, Ta...","[[], [ArmorSuit MilitaryShield Screen Protecto...",Samsung Galaxy Tab A 8.0 2017 Screen Protector...,[],[]
3252,3252,AFZUK3MTBIBEDQOPAK3OATUOUKLA,"[B07S8F43Y6, B07CPSZXLK, B07FTCRKB7, B076Q643S...",B07RTR39H6,5,"[2019-09-19T17:23:05.711000, 2019-09-25T00:59:...","[5, 2, 5, 5, 1]",2020-01-13 17:38:07.178,3,None,None,None,"[D-Line Cord Cover 2-Pack, Cord Hiders, Cable ...","[[Electronics, Accessories & Supplies, Cord Ma...","[[], [], [Tidy up your Home and Office, No mor...",KuToo Replacement for Fitbit Charge 3/Charge 4...,"[Electronics, Wearable Technology, Arm & Wrist...",[]
136,136,AE6LQC6BXTLVFJ2MTWAOPIDOSY6Q,"[B0BBMLD8QT, B003VAHYQY, B008J0Z9TA, B01EL0GY1...",B00CLZ047Q,5,"[2013-01-06T22:01:07.000000, 2013-01-06T22:03:...","[5, 5, 5, 5, 5]",2014-12-14 19:44:03.000,5,None,None,None,[Logitech S150 USB Speakers with Digital Sound...,"[[Electronics, Computers & Accessories, Comput...","[[There are plenty of speakers out there, with...",AOC I2267FW 22-Inch Class IPS Frameless/Slim L...,"[Electronics, Computers & Accessories, Monitors]","[Product description, I2267FW - 22? 16:9 LED L..."
2668,2668,AFQQQ5LGNSQUEBGDCYBAZZE5T3DA,"[B09RYVXYLV, B0848H7FKT, B0845NH17T, B094F8DLW...",B0BZHM9JD9,5,"[2020-12-10T18:55:23.588000, 2020-12-13T17:10:...","[5, 5, 5, 5, 5]",2020-12-29 00:28:49.884,5,None,None,None,[Arlo Ultra 2 Spotlight Camera - Add-on - Wire...,"[[Electronics, Camera & Photo, Video Surveilla...","[[Cutting edge protection, picture perfect. Zo...",StarTech.com USB-C & USB-A Dock - Hybrid Unive...,"[Electronics, Computers & Accessories, Laptop ...",[Dual 4K DisplayPort | Up to 4096 x 2160p @ 60Hz]


In [13]:
print("Missing history titles:",
      dataset["history_titles"].apply(lambda x: any(v is None for v in x)).sum())
print("Missing ground-truth titles:",
      dataset["ground_truth_title"].isna().sum())
print("History descriptions missing:",
      dataset["history_descriptions"].apply(
          lambda x: sum(
              (v is None) or (isinstance(v, list) and len(v) == 0)
              for v in x
          )
      ).sum())
print("Ground truth descriptions missing:",
      dataset["ground_truth_description"].apply(
          lambda x: isinstance(x, list) and len(x) == 0
      ).sum())

Missing history titles: 0
Missing ground-truth titles: 0
History descriptions missing: 12502
Ground truth descriptions missing: 2721


In [14]:
dataset.to_parquet(OUTPUT_PATH,index=False)
print("Saved enriched dataset to:")
print(OUTPUT_PATH)

Saved enriched dataset to:
/users/adhp511/query-conditioned-recommender/data/processed/enriched_user_histories.parquet
